In [1]:
!pip install --upgrade langchain langgraph langchain-google-genai langchain-community ddgs langchain_mcp_adapters mcp==1.29.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found exis

In [2]:
import os
from google.colab import userdata

os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "default"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "test"


In [3]:
from google.colab import userdata
from langchain.chat_models import init_chat_model

google_api_key = userdata.get('GOOGLE_API_KEY')

llm = init_chat_model("gemini-3.6-flash", model_provider="google_genai", api_key=google_api_key)


# MCP

In [4]:
import asyncio
import os

from langgraph.prebuilt import create_react_agent

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "huggingface": {
            "transport": "http",
            "url": "https://hf.co/mcp",
            }
        }
)

tools =  await client.get_tools()

for tool in tools:
  print(f" - {tool.name}: {tool.description[:100]}...")

 - hf_whoami: Inspect the current Hugging Face authentication context, including the account, visible organization...
 - hub_repo_search: Search Hugging Face repositories with a shared query interface. You can target models, datasets, spa...
 - hub_repo_details: Get details for one or more Hugging Face repos (model, dataset, or space). Auto-detects type unless ...
 - hf_fs: When to use: Hugging Face Hub models, datasets, Spaces, collections, papers, daily papers, today's t...


In [6]:
agent = create_react_agent(llm, tools)

query = "Can you search Hugging Face for the most popular text-to-image models?"
response = await agent.ainvoke({"messages": [("user", query)]})


/tmp/ipykernel_487/2085034343.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [7]:
print(response["messages"][-1].text)

Here are the most popular text-to-image models on Hugging Face, categorized by downloads and community likes:

---

### 🌟 Top Models by Likes & Downloads

#### 1. [black-forest-labs/FLUX.1-dev](https://hf.co/black-forest-labs/FLUX.1-dev)
* **Likes:** ~14.3K | **Downloads:** ~724K
* **Overview:** Currently the most liked text-to-image model on Hugging Face. Created by Black Forest Labs (ex-Stability AI founders), FLUX.1-dev offers state-of-the-art visual quality, complex prompt adherence, and realistic rendering.

#### 2. [stabilityai/stable-diffusion-xl-base-1.0](https://hf.co/stabilityai/stable-diffusion-xl-base-1.0)
* **Likes:** ~8.1K | **Downloads:** ~1.7M
* **Overview:** The cornerstone 1024x1024 base model from Stability AI. It remains one of the most widely used and fine-tuned open foundation models in AI art.

#### 3. [stable-diffusion-v1-5/stable-diffusion-v1-5](https://hf.co/stable-diffusion-v1-5/stable-diffusion-v1-5)
* **Likes:** ~1.2K | **Downloads:** ~1.5M
* **Overview:** 

In [8]:
from langchain_core.messages import AIMessage, HumanMessage

def print_messages(messages):
    for idx, msg in enumerate(messages):
        role = "AI" if isinstance(msg, AIMessage) else "User" if isinstance(msg, HumanMessage) else "Tool"
        tool_calls_str = ""
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_calls_str = f" Generates Tool Calls:"
            for tc in msg.tool_calls:
              tool_calls_str += f"\n     Tool: {tc['name']} Args: [{tc['args']}]"
        if len(msg.text) > 60:
          text = msg.text[:60] + "... <truncated>"
        else:
          text = msg.text
        print(f"  [{idx + 1}] {role}: {text}{tool_calls_str}")

In [9]:
print_messages(response["messages"])

  [1] User: Can you search Hugging Face for the most popular text-to-ima... <truncated>
  [2] AI:  Generates Tool Calls:
     Tool: hub_repo_search Args: [{'sort': 'downloads', 'filters': ['text-to-image'], 'repo_types': ['model'], 'limit': 10}]
  [3] Tool: Found 10 repositories across models matching filters [text-t... <truncated>
  [4] AI:  Generates Tool Calls:
     Tool: hub_repo_search Args: [{'sort': 'likes', 'filters': ['text-to-image'], 'limit': 10, 'repo_types': ['model']}]
  [5] Tool: Found 10 repositories across models matching filters [text-t... <truncated>
  [6] AI: Here are the most popular text-to-image models on Hugging Fa... <truncated>


In [10]:
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_current_date() -> str:
    """Returns the current date. Always use this to get the current date or year rather than searching or guessing."""
    return datetime.now().strftime("%Y-%m-%d")

In [11]:
agent2 = create_react_agent(llm, tools+[get_current_date])

query = "What is the current date?"
response = await agent2.ainvoke({"messages": [("user", query)]})


/tmp/ipykernel_487/3602350584.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent2 = create_react_agent(llm, tools+[get_current_date])


In [12]:
print_messages(response["messages"])

  [1] User: What is the current date?
  [2] AI:  Generates Tool Calls:
     Tool: get_current_date Args: [{}]
  [3] Tool: 2026-08-31
  [4] AI: Today's date is August 31, 2026.


# Retry policy

In [13]:
from langgraph.types import RetryPolicy
from requests.exceptions import ConnectionError
from langgraph.runtime import Runtime
from typing import TypedDict

class ApiState(TypedDict):
    query: str
    result: str

def call_unreliable_api(state: ApiState, runtime: Runtime):
    print(f"\n[Node: call_unreliable_api] Attempt {runtime.execution_info.node_attempt}")

    if runtime.execution_info.node_attempt < 3:
        print("  -> Simulation: Network error occurred! Raising ConnectionError.")
        raise ConnectionError("Simulated transient network error.")

    print("  -> Simulation: API call succeeded!")
    return {"result": f"Data for {state['query']}"}

def process_data(state: ApiState):
    print("\n[Node: process_data] Processing the retrieved data...")
    return {"result": state["result"] + " (processed)"}

In [14]:
from langgraph.graph import StateGraph, START, END

retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=0.5,
    backoff_factor=2.0,
    retry_on=ConnectionError
)

builder = StateGraph(ApiState)
builder.set_node_defaults(retry_policy=retry_policy)

builder.add_node(
    "call_unreliable_api",
    call_unreliable_api,
    retry_policy=retry_policy
)
builder.add_node("process_data", process_data)

builder.add_edge(START, "call_unreliable_api")
builder.add_edge("call_unreliable_api", "process_data")
builder.add_edge("process_data", END)

graph = builder.compile()

In [15]:
inputs = {"query": "LangGraph retry example"}
final_state = graph.invoke(inputs)
print(f"Final State: {final_state}")


[Node: call_unreliable_api] Attempt 1
  -> Simulation: Network error occurred! Raising ConnectionError.

[Node: call_unreliable_api] Attempt 2
  -> Simulation: Network error occurred! Raising ConnectionError.

[Node: call_unreliable_api] Attempt 3
  -> Simulation: API call succeeded!

[Node: process_data] Processing the retrieved data...
Final State: {'query': 'LangGraph retry example', 'result': 'Data for LangGraph retry example (processed)'}


In [16]:
import asyncio
from langgraph.errors import NodeTimeoutError

class State(TypedDict):
    task: str
    result: str

async def slow_node(state: State, runtime: Runtime):
    print(f"\n[Node: slow_node] Starting task: {state['task']}")
    if runtime.execution_info.node_attempt < 2:
      await asyncio.sleep(3)
    print("  -> Task completed successfully!")
    return {"result": f"Completed: {state['task']}"}

async def fast_node(state: State):
    print(f"\n[Node: fast_node] Fast processing...")
    return {"result": f"Fast result for: {state['task']}"}



In [17]:
from langgraph.types import TimeoutPolicy

builder = StateGraph(State)

retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=0.5,
    backoff_factor=2.0,
)

builder.add_node(
    "slow_node",
    slow_node,
    timeout=TimeoutPolicy(run_timeout=1.0, idle_timeout=2.0),
    retry_policy=retry_policy
)
builder.add_node("fast_node", fast_node)

builder.add_edge(START, "slow_node")
builder.add_edge("slow_node", "fast_node")
builder.add_edge("fast_node", END)

graph = builder.compile()

In [18]:
inputs = {"task": "LangGraph timeout example"}
final_state = await graph.ainvoke(inputs)



[Node: slow_node] Starting task: LangGraph timeout example

[Node: slow_node] Starting task: LangGraph timeout example
  -> Task completed successfully!

[Node: fast_node] Fast processing...


In [19]:
builder = StateGraph(State)

builder.add_node(
    "slow_node",
    slow_node,
    timeout=TimeoutPolicy(run_timeout=4.0, idle_timeout=4.0)
)
builder.add_node("fast_node", fast_node)

builder.add_edge(START, "slow_node")
builder.add_edge("slow_node", "fast_node")
builder.add_edge("fast_node", END)

graph = builder.compile()

In [20]:
inputs = {"task": "LangGraph timeout example"}
final_state = await graph.ainvoke(inputs)


[Node: slow_node] Starting task: LangGraph timeout example
  -> Task completed successfully!

[Node: fast_node] Fast processing...


In [21]:
from langchain_core.language_models import SimpleChatModel

global call_count
call_count = 0

class FailingChatModel(SimpleChatModel):
    """Simulates an unreliable LLM that fails on the first two attempts."""

    def _call(self, messages, stop=None, run_manager=None, **kwargs) -> str:
        global call_count
        call_count += 1
        print(f"  [Primary Model] Attempt {call_count}")
        if call_count < 3:
            raise ConnectionError("Simulated transient connection failure.")
        return "Response from Primary Model!"

    @property
    def _llm_type(self) -> str:
        return "failing-chat"

class BackupChatModel(SimpleChatModel):
    """Simulates a reliable fallback LLM."""

    def _call(self, messages, stop=None, run_manager=None, **kwargs) -> str:
        print("  [Backup Model] Invoked successfully!")
        return "Response from Backup Model!"

    @property
    def _llm_type(self) -> str:
        return "backup-chat"

In [22]:
primary_model = FailingChatModel()
backup_model = BackupChatModel()

call_count = 0

resilient_model = primary_model.with_retry(
  stop_after_attempt=3,
  retry_if_exception_type=(ConnectionError,)
)
response = resilient_model.invoke("Hello")
print(f"Result: {response.text}\n")


  [Primary Model] Attempt 1
  [Primary Model] Attempt 2
  [Primary Model] Attempt 3
Result: Response from Primary Model!



In [23]:
call_count = 0

fallback_model = primary_model.with_retry(stop_after_attempt=1).with_fallbacks(
    fallbacks=[backup_model],
    exceptions_to_handle=(ConnectionError,)
)
response = fallback_model.invoke("Hello")
print(f"Result: {response.text}")

  [Primary Model] Attempt 1
  [Backup Model] Invoked successfully!
Result: Response from Backup Model!


## Error handling

In [24]:
from langgraph.errors import NodeError
from langgraph.types import Command

def api_error_handler(state: ApiState, error: NodeError) -> Command:
    print(f"\n[Error Handler] Caught error from '{error.node}': {error.error}")
    return Command(
        update={"result": "Fallback API response"},
        goto="process_data"
    )

retry_policy = RetryPolicy(
    max_attempts=2,
    initial_interval=0.5,
    backoff_factor=2.0,
    retry_on=ConnectionError
)

builder = StateGraph(ApiState)
builder.add_node(
    "call_unreliable_api",
    call_unreliable_api,
    retry_policy=retry_policy,
    error_handler=api_error_handler
)
builder.add_node("process_data", process_data)

builder.add_edge(START, "call_unreliable_api")
builder.add_edge("call_unreliable_api", "process_data")
builder.add_edge("process_data", END)

graph = builder.compile()

In [25]:
inputs = {"query": "LangGraph error handler example", "result": ""}
final_state = graph.invoke(inputs)
print(f"Final State: {final_state}")


[Node: call_unreliable_api] Attempt 1
  -> Simulation: Network error occurred! Raising ConnectionError.

[Node: call_unreliable_api] Attempt 2
  -> Simulation: Network error occurred! Raising ConnectionError.

[Error Handler] Caught error from 'call_unreliable_api': Simulated transient network error.

[Node: process_data] Processing the retrieved data...
Final State: {'query': 'LangGraph error handler example', 'result': 'Fallback API response (processed)'}
